# Free generation and natural-span patching
Choose an A100 runtime. Read docs/free_generation.md. This code-only notebook never uploads results. Preserve artifacts before rerunning or releasing the runtime.

In [ ]:
import pathlib, subprocess, sys
REPO = "https://github.com/Twist-Shan/DLM_Multitask_Reasoning.git"
REVISION = "main"  # Replace with an exact commit SHA for reproducibility.
ROOT = pathlib.Path("/content/DLM_Multitask_Reasoning")
if ROOT.exists():
    raise RuntimeError("Workspace already exists; preserve prior outputs before a new run.")
subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
subprocess.run(["git", "checkout", REVISION], cwd=ROOT, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-r", "requirements/models.txt", "-r", "requirements/analysis.txt"], cwd=ROOT, check=True)
subprocess.run(["nvidia-smi"], check=True)
def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)
run("-m", "unittest", "discover", "-s", "tests", "-v")


In [ ]:
run("scripts/run_free_reasoning.py")
run("scripts/continue_free_reasoning.py")

Inspect generated responses first. Patching requires explicit arithmetic spans and a For B: section; unsupported wording is reported, not repaired.

In [ ]:
run("scripts/run_free_reasoning_patching.py")
run("scripts/trace_free_patching.py", "--run", "runs/free-reasoning-patching")

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/free_reasoning_artifacts", "zip", ROOT / "runs")
files.download(archive)